In [1]:
years_to_process = "2015,2016,2017"

StatementMeta(, db6808ab-b185-4485-ba31-baef261e2c0d, 3, Finished, Available, Finished, False)

In [2]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType
)

YEARS = [int(y) for y in years_to_process.split(",")]

BRONZE_ONTIME = "Files/bronze/on_time"
BRONZE_T100   = "Files/bronze/t100"

# We declare only the ~22 columns we use. The BTS file has 100+.
# Reading fewer columns is a real performance win.
#
# FlightDate is StringType deliberately: date parsing failures return null
# silently, so we cast explicitly later and can COUNT the failures.


t100_schema = StructType([
    StructField("YEAR",           IntegerType()),
    StructField("MONTH",          IntegerType()),
    StructField("UNIQUE_CARRIER", StringType()),
    StructField("CARRIER_NAME",   StringType()),
    StructField("ORIGIN",         StringType()),
    StructField("DEST",           StringType()),
    StructField("AIRCRAFT_TYPE",  StringType()),
    StructField("DEPARTURES_PERFORMED", DoubleType()),
    StructField("PAYLOAD",        DoubleType()),
    StructField("FREIGHT",        DoubleType()),
    StructField("MAIL",           DoubleType()),
    StructField("DISTANCE",       DoubleType()),
    StructField("SEATS",          DoubleType()),
])

print(f"Processing years: {YEARS}")

StatementMeta(, db6808ab-b185-4485-ba31-baef261e2c0d, 4, Finished, Available, Finished, False)

Processing years: [2015, 2016, 2017]


In [3]:
# Only read year folders that actually exist. A glob tolerates missing months
# inside a year, but a missing YEAR folder fails the whole read — so we probe
# first. This keeps the notebook runnable mid-backfill.
paths = []
for y in YEARS:
    year_path = f"{BRONZE_ONTIME}/year={y}"
    try:
        mssparkutils.fs.ls(year_path)
        paths.append(f"{year_path}/month=*/on_time.csv")
        print(f"  found year={y}")
    except Exception:
        print(f"  (no data yet for year={y} — skipping)")

if not paths:
    raise ValueError("No bronze data found. Run nb_ingest_bronze first.")

# NO schema here: Spark maps a supplied schema by POSITION, and the BTS file
# has 100+ columns in an order we don't control — a positional schema silently
# mis-assigns every column. Read all-string, then select BY NAME in Cell 4.
df_raw = (spark.read
    .option("header", True)
    .option("mode", "PERMISSIVE")   # bad rows -> nulls, don't kill the job
    .csv(paths))

print(f"\nRaw rows read: {df_raw.count():,}")
print(f"Columns available: {len(df_raw.columns)}")

# Fail loudly if BTS ever renames something we depend on.
required = ["FlightDate", "Reporting_Airline", "Flight_Number_Reporting_Airline",
            "Origin", "Dest", "DepDelay", "ArrDelay", "Cancelled",
            "Diverted", "Distance"]
missing = [c for c in required if c not in df_raw.columns]
if missing:
    raise ValueError(f"Expected columns missing from source: {missing}")
print("All required columns present.")

StatementMeta(, db6808ab-b185-4485-ba31-baef261e2c0d, 5, Finished, Available, Finished, False)

  found year=2015
  (no data yet for year=2016 — skipping)
  (no data yet for year=2017 — skipping)

Raw rows read: 1,403,471
Columns available: 110
All required columns present.


In [4]:
# Everything arrived as string. Cast explicitly, BY NAME.
df_typed = df_raw.select(
    F.col("FlightDate"),
    F.col("Reporting_Airline"),
    F.col("Flight_Number_Reporting_Airline"),
    F.col("Tail_Number"),
    F.col("Origin"),
    F.col("Dest"),
    F.col("DepDelay").cast("double").alias("DepDelay"),
    F.col("ArrDelay").cast("double").alias("ArrDelay"),
    F.col("Cancelled").cast("double").alias("Cancelled"),
    F.col("Diverted").cast("double").alias("Diverted"),
    F.col("Distance").cast("double").alias("Distance"),
    F.col("CarrierDelay").cast("double").alias("CarrierDelay"),
    F.col("WeatherDelay").cast("double").alias("WeatherDelay"),
    F.col("NASDelay").cast("double").alias("NASDelay"),
    F.col("SecurityDelay").cast("double").alias("SecurityDelay"),
    F.col("LateAircraftDelay").cast("double").alias("LateAircraftDelay"),
)


df_clean = (df_typed
    .withColumn("flight_date", F.to_date(F.col("FlightDate"), "yyyy-MM-dd"))

    # Trim + uppercase: 'DL ' and 'dl' must become 'DL', or dimension joins
    # silently produce duplicates.
    .withColumn("carrier", F.upper(F.trim(F.col("Reporting_Airline"))))
    .withColumn("origin",  F.upper(F.trim(F.col("Origin"))))
    .withColumn("dest",    F.upper(F.trim(F.col("Dest"))))
    .withColumn("route",   F.concat_ws("->", F.upper(F.trim(F.col("Origin"))),
                                             F.upper(F.trim(F.col("Dest")))))

    # Keep a flag so we can distinguish "genuinely 0" from "was missing".
    .withColumn("arr_delay_missing", F.col("ArrDelay").isNull().cast("int"))
    .withColumn("arr_delay", F.coalesce(F.col("ArrDelay"), F.lit(0.0)))
    .withColumn("dep_delay", F.coalesce(F.col("DepDelay"), F.lit(0.0)))

    .withColumn("cancelled", F.coalesce(F.col("Cancelled"), F.lit(0.0)).cast("int"))
    .withColumn("diverted",  F.coalesce(F.col("Diverted"),  F.lit(0.0)).cast("int"))
    .withColumn("distance_miles", F.col("Distance"))

    # BTS industry standard: on time = arrives LESS THAN 15 min late.
    # Cancelled flights are neither on-time nor delayed — they're null.
    .withColumn("on_time",
        F.when(F.col("Cancelled") == 1, None)
         .when(F.coalesce(F.col("ArrDelay"), F.lit(0.0)) < 15, 1)
         .otherwise(0))
    .withColumn("delayed",
        F.when(F.col("Cancelled") == 1, None)
         .when(F.coalesce(F.col("ArrDelay"), F.lit(0.0)) >= 15, 1)
         .otherwise(0))
)

print("Cleaned. Sample:")
df_clean.select("flight_date","carrier","route","arr_delay","on_time").show(5)

StatementMeta(, db6808ab-b185-4485-ba31-baef261e2c0d, 6, Finished, Available, Finished, False)

Cleaned. Sample:
+-----------+-------+--------+---------+-------+
|flight_date|carrier|   route|arr_delay|on_time|
+-----------+-------+--------+---------+-------+
| 2015-01-01|     AA|JFK->LAX|      7.0|      1|
| 2015-01-02|     AA|JFK->LAX|    -19.0|      1|
| 2015-01-03|     AA|JFK->LAX|    -39.0|      1|
| 2015-01-04|     AA|JFK->LAX|    -12.0|      1|
| 2015-01-05|     AA|JFK->LAX|     -8.0|      1|
+-----------+-------+--------+---------+-------+
only showing top 5 rows



In [5]:
cause_cols = {
    "Carrier":       "CarrierDelay",
    "Weather":       "WeatherDelay",
    "NAS":           "NASDelay",
    "Security":      "SecurityDelay",
    "Late Aircraft": "LateAircraftDelay",
}

# BTS splits delay minutes across five causes. Pick the LARGEST as primary —
# one categorical column is far more usable in a dashboard than five numerics.
greatest_delay = F.greatest(*[F.coalesce(F.col(c), F.lit(0.0))
                              for c in cause_cols.values()])

cause_expr = (F.when(F.col("cancelled") == 1, F.lit("Cancelled"))
               .when(greatest_delay <= 0, F.lit("On-Time")))
for label, colname in cause_cols.items():
    cause_expr = cause_expr.when(
        F.coalesce(F.col(colname), F.lit(0.0)) == greatest_delay, F.lit(label))

df_clean = (df_clean
    .withColumn("delay_cause", cause_expr.otherwise(F.lit("Unknown")))
    .withColumn("carrier_delay_min",       F.coalesce(F.col("CarrierDelay"), F.lit(0.0)))
    .withColumn("weather_delay_min",       F.coalesce(F.col("WeatherDelay"), F.lit(0.0)))
    .withColumn("nas_delay_min",           F.coalesce(F.col("NASDelay"), F.lit(0.0)))
    .withColumn("security_delay_min",      F.coalesce(F.col("SecurityDelay"), F.lit(0.0)))
    .withColumn("late_aircraft_delay_min", F.coalesce(F.col("LateAircraftDelay"), F.lit(0.0)))
)

StatementMeta(, db6808ab-b185-4485-ba31-baef261e2c0d, 7, Finished, Available, Finished, False)

In [6]:
# These were DAX calculated columns in your Power BI model.
# DirectLake doesn't support calculated columns, so they move upstream —
# computed once at write time, available to every consumer, testable here.
df_clean = (df_clean
    .withColumn("delay_severity_band",
        F.when(F.col("cancelled") == 1,   "Cancelled")
         .when(F.col("arr_delay") < 15,   "On Time (<15 min)")
         .when(F.col("arr_delay") <= 45,  "Minor (15-45 min)")
         .when(F.col("arr_delay") <= 120, "Major (46-120 min)")
         .otherwise("Severe (120+ min)"))

    # Text sorts alphabetically, which would put "Severe" before "On Time".
    # An integer companion column fixes ordering in every visual.
    .withColumn("delay_severity_sort",
        F.when(F.col("cancelled") == 1,   0)
         .when(F.col("arr_delay") < 15,   1)
         .when(F.col("arr_delay") <= 45,  2)
         .when(F.col("arr_delay") <= 120, 3)
         .otherwise(4))

    # Departed late but arrived on time — schedule padding doing real work.
    .withColumn("recovered_in_flight",
        F.when((F.col("dep_delay") > 15) & (F.col("arr_delay") < 15), 1)
         .otherwise(0))
)

StatementMeta(, db6808ab-b185-4485-ba31-baef261e2c0d, 8, Finished, Available, Finished, False)

In [7]:
before = df_clean.count()

# Natural key for a flight. BTS republishes months with corrections, so rows
# can differ byte-wise while representing the SAME flight. Whole-row dedupe
# would miss those and double-count.
df_silver = df_clean.dropDuplicates([
    "flight_date", "carrier",
    "Flight_Number_Reporting_Airline", "origin", "dest"
])
after = df_silver.count()
print(f"Dedupe: {before:,} -> {after:,}  ({before - after:,} removed)")

df_silver_final = df_silver.select(
    "flight_date",
    F.year("flight_date").alias("year"),
    F.month("flight_date").alias("month"),
    "carrier", "origin", "dest", "route",
    F.col("Flight_Number_Reporting_Airline").alias("flight_number"),
    F.col("Tail_Number").alias("tail_number"),
    "dep_delay", "arr_delay", "arr_delay_missing",
    "cancelled", "diverted", "distance_miles",
    "on_time", "delayed", "delay_cause",
    "carrier_delay_min", "weather_delay_min", "nas_delay_min",
    "security_delay_min", "late_aircraft_delay_min",
    "delay_severity_band", "delay_severity_sort", "recovered_in_flight",
).filter(F.col("flight_date").isNotNull())

# Partition by year/month because that's how the data is QUERIED.
# Partition on the filter column, not on what's convenient to write.
(df_silver_final.write
    .format("delta").mode("overwrite")
    .partitionBy("year", "month")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_flights"))

print(f"silver_flights written: {df_silver_final.count():,} rows")

StatementMeta(, db6808ab-b185-4485-ba31-baef261e2c0d, 9, Finished, Available, Finished, False)

Dedupe: 1,403,471 -> 1,403,471  (0 removed)
silver_flights written: 1,403,471 rows


In [8]:
print("=" * 60)
print("SILVER DATA QUALITY")
print("=" * 60)

spark.sql("""
    SELECT year, month,
           COUNT(*)                                                AS flights,
           SUM(cancelled)                                          AS cancelled,
           ROUND(AVG(arr_delay), 2)                                AS avg_arr_delay,
           ROUND(100.0 * SUM(on_time) / SUM(on_time + delayed), 1) AS otp_pct
    FROM silver_flights
    GROUP BY year, month ORDER BY year, month
""").show(50)

spark.sql("""
    SELECT delay_severity_band, delay_severity_sort, COUNT(*) AS flights
    FROM silver_flights
    GROUP BY delay_severity_band, delay_severity_sort
    ORDER BY delay_severity_sort
""").show()

StatementMeta(, db6808ab-b185-4485-ba31-baef261e2c0d, 10, Finished, Available, Finished, False)

SILVER DATA QUALITY
+----+-----+-------+---------+-------------+-------+
|year|month|flights|cancelled|avg_arr_delay|otp_pct|
+----+-----+-------+---------+-------------+-------+
|2015|    1| 469968|    11982|         5.65|   79.0|
|2015|    2| 429191|    20517|          7.9|   76.7|
|2015|    3| 504312|    11002|          4.8|   80.7|
+----+-----+-------+---------+-------------+-------+

+-------------------+-------------------+-------+
|delay_severity_band|delay_severity_sort|flights|
+-------------------+-------------------+-------+
|          Cancelled|                  0|  43501|
|  On Time (<15 min)|                  1|1073388|
|  Minor (15-45 min)|                  2| 170725|
| Major (46-120 min)|                  3|  86984|
|  Severe (120+ min)|                  4|  28873|
+-------------------+-------------------+-------+



In [9]:
# --- find the T-100 files -------------------------------------------------
# Glob rather than exact filename: BTS names them
# T_T100_SEGMENT_US_CARRIER_ONLY_<year>.csv, not what we'd have chosen.
t100_paths = []
for y in YEARS:
    folder = f"{BRONZE_T100}/year={y}"
    try:
        files = mssparkutils.fs.ls(folder)
        if any(f.name.lower().endswith(".csv") for f in files):
            t100_paths.append(f"{folder}/*.csv")
            print(f"  found T-100 for {y}")
    except Exception:
        print(f"  (no T-100 folder for {y})")

if not t100_paths:
    raise ValueError("No T-100 files found. Upload to Files/bronze/t100/year=<year>/")

# --- read all-string, select BY NAME --------------------------------------
# Read without a supplied schema: a schema maps by POSITION, and this file's
# column order is not what we'd assume. basePath makes Spark expose the
# year=<n> folder as a partition column — the file has MONTH but no YEAR,
# because each download covers a single year.
df_t100_raw = (spark.read
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .option("basePath", BRONZE_T100)
    .csv(t100_paths))

print(f"\nT-100 raw rows: {df_t100_raw.count():,}")
print(f"Columns: {df_t100_raw.columns}")

required = ["MONTH", "UNIQUE_CARRIER", "ORIGIN", "DEST",
            "FREIGHT", "MAIL", "PAYLOAD", "DISTANCE", "DEPARTURES_PERFORMED"]
missing = [c for c in required if c not in df_t100_raw.columns]
if missing:
    raise ValueError(f"Expected T-100 columns missing: {missing}")
print("All required columns present.")

# --- clean, convert units, derive FTK -------------------------------------
df_t100 = (df_t100_raw
    .withColumn("year",  F.col("year").cast("int"))     # from the partition folder
    .withColumn("month", F.col("MONTH").cast("int"))
    .withColumn("carrier", F.upper(F.trim(F.col("UNIQUE_CARRIER"))))
    .withColumn("origin",  F.upper(F.trim(F.col("ORIGIN"))))
    .withColumn("dest",    F.upper(F.trim(F.col("DEST"))))
    .withColumn("route",   F.concat_ws("->", F.upper(F.trim(F.col("ORIGIN"))),
                                             F.upper(F.trim(F.col("DEST")))))
    .withColumn("carrier_name",  F.col("CARRIER_NAME"))
    .withColumn("aircraft_type", F.col("AIRCRAFT_TYPE"))

    # T-100 reports POUNDS and MILES. Convert once here so no downstream
    # consumer can mix units — silent unit errors are expensive.
    .withColumn("freight_tonnes",  F.col("FREIGHT").cast("double") / 2204.62)
    .withColumn("mail_tonnes",     F.col("MAIL").cast("double") / 2204.62)
    .withColumn("capacity_tonnes", F.col("PAYLOAD").cast("double") / 2204.62)
    .withColumn("distance_km",     F.col("DISTANCE").cast("double") * 1.60934)
    .withColumn("departures",      F.col("DEPARTURES_PERFORMED").cast("double"))

    # FTK = freight tonne-km: the industry measure of cargo work done.
    .withColumn("ftk",  F.col("freight_tonnes")  * F.col("distance_km"))
    .withColumn("aftk", F.col("capacity_tonnes") * F.col("distance_km"))

    # T-100 is monthly with no day. Anchor to the 1st so it joins to a date
    # dimension. Deliberate and documented, not a guess.
    .withColumn("flight_date",
        F.to_date(F.concat_ws("-", F.col("year"), F.col("month"), F.lit("01"))))
)

df_t100_final = (df_t100.select(
    "flight_date", "year", "month",
    "carrier", "carrier_name", "origin", "dest", "route",
    "aircraft_type", "departures",
    "freight_tonnes", "mail_tonnes", "capacity_tonnes",
    "distance_km", "ftk", "aftk",
).filter(F.col("flight_date").isNotNull()))

(df_t100_final.write
    .format("delta").mode("overwrite")
    .partitionBy("year", "month")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_tonnage"))

print(f"\nsilver_tonnage written: {df_t100_final.count():,} rows")

StatementMeta(, db6808ab-b185-4485-ba31-baef261e2c0d, 11, Finished, Available, Finished, False)

  found T-100 for 2015
  found T-100 for 2016
  found T-100 for 2017

T-100 raw rows: 1,272,672
Columns: ['DEPARTURES_SCHEDULED', 'DEPARTURES_PERFORMED', 'PAYLOAD', 'SEATS', 'PASSENGERS', 'FREIGHT', 'MAIL', 'DISTANCE', 'RAMP_TO_RAMP', 'AIR_TIME', 'UNIQUE_CARRIER', 'UNIQUE_CARRIER_NAME', 'CARRIER_NAME', 'ORIGIN_AIRPORT_ID', 'ORIGIN', 'ORIGIN_CITY_NAME', 'ORIGIN_STATE_ABR', 'ORIGIN_STATE_NM', 'DEST_AIRPORT_ID', 'DEST', 'DEST_CITY_NAME', 'DEST_STATE_ABR', 'DEST_STATE_NM', 'AIRCRAFT_TYPE', 'MONTH', 'year']
All required columns present.

silver_tonnage written: 1,272,672 rows


In [10]:
spark.sql("""
    SELECT year,
           COUNT(*)                               AS segments,
           COUNT(DISTINCT carrier)                AS carriers,
           ROUND(SUM(freight_tonnes), 0)          AS freight_tonnes,
           ROUND(100.0 * SUM(ftk) / SUM(aftk), 1) AS load_factor_pct
    FROM silver_tonnage
    GROUP BY year ORDER BY year
""").show()

StatementMeta(, db6808ab-b185-4485-ba31-baef261e2c0d, 12, Finished, Available, Finished, False)

+----+--------+--------+--------------+---------------+
|year|segments|carriers|freight_tonnes|load_factor_pct|
+----+--------+--------+--------------+---------------+
|2015|  418800|     126|   1.3691673E7|           15.1|
|2016|  431622|     128|   1.4164421E7|           14.8|
|2017|  422250|     126|   1.5224556E7|           15.6|
+----+--------+--------+--------------+---------------+



In [11]:
spark.sql("SHOW TABLES").show()
print("silver_flights rows:", spark.table("silver_flights").count())

StatementMeta(, db6808ab-b185-4485-ba31-baef261e2c0d, 13, Finished, Available, Finished, False)

+--------------------+--------------+-----------+
|           namespace|     tableName|isTemporary|
+--------------------+--------------+-----------+
|`Cargo-Ops-Analyt...|pipeline_state|      false|
|`Cargo-Ops-Analyt...|silver_flights|      false|
|`Cargo-Ops-Analyt...|silver_tonnage|      false|
+--------------------+--------------+-----------+

silver_flights rows: 1403471
